# Inventory: Bronze -> Silver

Drop bad rows, normalize types and text.

In [1]:
%run ../00_config.ipynb
%run ../00_utils.ipynb

/usr/local/lib/python3.12/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


[08/23/26 15:05:57] INFO     Using                                                                  ]8;id=11848932;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=11848933;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py#302\302]8;;\
                             '/usr/local/lib/python3.12/site-packages/kedro/framework/project/rich_                
                             logging.yml' as logging configuration.                                                

[08/23/26 15:05:57] WARNING  /usr/local/lib/python3.12/site-packages/kedro/framework/context/contex ]8;id=11848940;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=11848941;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             t.py:221: UserWarning: Parameters not found in your Kedro project                     
                             config.                                                                               
                             No files of YAML or JSON format found in /app/conf/base or                            
                             /app/conf/local matching the glob pattern(s): ['parameters*',                         
                             'parameters*/**', '**/parameters*']                                                   
                               warn(f"Parameters not found in your Kedro project config.\n{exc!s}")                
                                                                                                                   

                    INFO     No typed parameter requirements found, returning original   ]8;id=11848948;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py\parameter_validator.py]8;;\:]8;id=11848949;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py#124\124]8;;\
                             parameters                                                                            

Kedro context loaded from /app
Catalog datasets: ['raw_employees', 'bronze_employees', 'silver_employees', 'gold_employees', 'raw_sales', 'bronze_sales', 'silver_sales', 'gold_sales', 'raw_inventory', 'bronze_inventory', 'silver_inventory', 'gold_inventory', 'parameters']


                    WARNING  /usr/local/lib/python3.12/site-packages/nbformat/validator.py:434:     ]8;id=11848954;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=11848955;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             MissingIDFieldWarning: Cell is missing an id field, this will become a                
                             hard error in future nbformat versions. You may want to use                           
                             `normalize()` on your notebooks before validations (available since                   
                             nbformat 5.1.4). Previous versions of nbformat are fixing this issue                  
                             transparently, and will stop doing so in the future.                                  
                               _validate(nbdict, ref, version, version_minor, relax_add_props)                     
                                                                                                                   

In [2]:
bronze_inventory = catalog.load("bronze_inventory")
bronze_inventory

                    INFO     Loading data from bronze_inventory (CSVDataset)...                ]8;id=11848962;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=11848963;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1050\1050]8;;\

,item_id,item_name,category,stock_count,warehouse,last_updated,notes
0,201.0,Steel Bolt,hardware,500,WH-1,2023-01-10,restocked
1,202.0,Copper Wire,ELECTRICAL,150,WH-2,2023-02-15,NaN
2,203.0,Rubber Gasket,hardware,300,WH-1,2023-03-01,low stock
3,NaN,Ghost Item,unknown,0,WH-0,2022-01-01,"bad data, should be dropped"
4,204.0,LED Bulb,electrical,1000,WH-3,2023-04-20,NaN


In [3]:
import pandas as pd

silver_inventory = bronze_inventory.dropna(subset=["item_id"]).copy()
silver_inventory["item_id"] = silver_inventory["item_id"].astype(int)
silver_inventory["item_name"] = clean_text(silver_inventory["item_name"])
silver_inventory["category"] = clean_text(silver_inventory["category"]).str.lower()
silver_inventory["last_updated"] = pd.to_datetime(silver_inventory["last_updated"]).dt.date
silver_inventory

,item_id,item_name,category,stock_count,warehouse,last_updated,notes
0,201,Steel Bolt,hardware,500,WH-1,2023-01-10,restocked
1,202,Copper Wire,electrical,150,WH-2,2023-02-15,NaN
2,203,Rubber Gasket,hardware,300,WH-1,2023-03-01,low stock
4,204,LED Bulb,electrical,1000,WH-3,2023-04-20,NaN


In [4]:
catalog.save("silver_inventory", silver_inventory)

                    INFO     Saving data to silver_inventory (CSVDataset)...                   ]8;id=11848969;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=11848970;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1006\1006]8;;\

## PySpark alternative (reference only)

PySpark isn't installed in this image. Left commented out to show how this
stage would transform with Spark instead of pandas.

In [5]:
# from pyspark.sql.functions import col, lower, to_date, trim
#
# bronze_inventory_spark = (
#     spark.read.option("header", "true")
#     .csv(str(PROJECT_ROOT / "data/02_bronze/inventory.csv"))
# )
#
# silver_inventory_spark = (
#     bronze_inventory_spark
#     .filter(col("item_id").isNotNull())
#     .withColumn("item_id", col("item_id").cast("int"))
#     .withColumn("item_name", trim(col("item_name")))
#     .withColumn("category", lower(trim(col("category"))))
#     .withColumn("last_updated", to_date(col("last_updated")))
# )
#
# silver_inventory_spark.write.mode("overwrite").option("header", "true").csv(
#     str(PROJECT_ROOT / "data/03_silver/inventory.csv")
# )